# MacroRegimes3 — Part 3: Proxy-Variable Random Forest Regime Classification and Forecaster

Based on:
> Yuan Zhao, Xue Gong, Weiguo Zhang, Weijun Xu. *Stock return forecasting based on the proxy variables of category factors* (2025).

- A Random Forest classifier trained on the Bayesian-tree regimes gives per-regime SHAP explanations of which variables drive each regime.
- The ~60-variable feature space is compressed into 12 economically interpretable **proxy variables** (one per category)
  using SHAP importance weights from a Random Forest regressor targeting the NSS yield curve, and benchmarked against PCA and RMT-PCA.
- Expanding-window out-of-sample evaluation: contemporaneous classification (with and without bond-market variables),
  a 1-month-ahead forecaster, transition analysis and SHAP rank-stability checks.

Split out of `MacroRegimes2.ipynb`, Sections 11–15.

**Change vs MacroRegimes2 — sign-corrected proxy variables.** Proxy variables used to be |SHAP|-weighted averages of
quantile ranks, so counter-cyclical and pro-cyclical variables (e.g. jobless claims and job openings) were averaged with the
same sign and cancelled each other. Each variable is now **oriented** first: if higher values of the variable push the
predicted yield curve *down* (negative feature–SHAP covariance in the training window), its quantile is flipped to
`1 − q`. Every proxy variable therefore reads "high = pushes yields up". Orientation is re-estimated inside every OOS fold.

**Inputs:** `MacroRegimes3_Data_*.csv` (from `Data.ipynb`), `MacroRegimes3_Tree_Labels.csv`, `_Tree_FoldLabels.csv`,
`_Tree_FoldSplits.csv` (from `TreeClassifier.ipynb`).

**Outputs:**

| File | Contents | Used by |
|---|---|---|
| `MacroRegimes3_PVRF_ProxyVariables.csv` | 12 sign-corrected proxy variables (full-sample weights, all features) | HMM |
| `MacroRegimes3_PVRF_ProxyVariables_xbond.csv` | 11 proxy variables built without bond-market variables | HMM |
| `MacroRegimes3_PVRF_ProxyWeights.csv` | per-variable SHAP importance, orientation sign and weight | diagnostics |
| `MacroRegimes3_PVRF_OOS_Results.csv`, `_OOS_FullTruth_Results.csv`, `_OOS_xbond_Results.csv`, `_FCAST_Results.csv` | monthly OOS predictions | — |

## 0. Imports and User Inputs

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

try:
    import shap
except ImportError:
    !pip install shap -q
    import shap

try:
    from arch.bootstrap import MCS
except ImportError:
    !pip install arch -q
    from arch.bootstrap import MCS

# Folder holding every CSV input and output (locally: DataAndResults/ next to the MacroRegimes3/ notebook folder)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    data_folder = '/content/drive/MyDrive/DataAndResults'
except ImportError:
    data_folder = os.path.join('..', 'DataAndResults')

In [ ]:
n_trees = 100         # trees per random forest
random_state = 42
min_training_years = 5  # must match TreeClassifier.ipynb
gap_months = 1          # must match TreeClassifier.ipynb
max_regimes = 3
global_classes = [0, 1, 2]  # canonical regime labels

regime_colors = {0: 'coral', 1: 'steelblue', 2: 'mediumseagreen', 3: 'orchid', 4: 'goldenrod'}

In [ ]:
# Load data (Data.ipynb) and tree results (TreeClassifier.ipynb)
data_q      = pd.read_csv(f'{data_folder}/MacroRegimes3_Data_Quantiles.csv', index_col='date', parse_dates=True)
target      = pd.read_csv(f'{data_folder}/MacroRegimes3_Data_Target.csv', index_col='date', parse_dates=True)
recession   = pd.read_csv(f'{data_folder}/MacroRegimes3_Data_Recession.csv', index_col='date', parse_dates=True)
columns     = pd.read_csv(f'{data_folder}/MacroRegimes3_Data_Columns.csv')
tree_labels = pd.read_csv(f'{data_folder}/MacroRegimes3_Tree_Labels.csv', index_col='date', parse_dates=True)
fold_labels = pd.read_csv(f'{data_folder}/MacroRegimes3_Tree_FoldLabels.csv', parse_dates=['date'])
fold_splits = pd.read_csv(f'{data_folder}/MacroRegimes3_Tree_FoldSplits.csv')

common_idx = data_q.index
all_cols = columns['variable'].tolist()
bond_cols = columns.loc[columns['group'] == 'bond', 'variable'].tolist()
categories = {cat: g['variable'].tolist() for cat, g in columns.groupby('pv_category', sort=False)}
feature_sets = {'all': all_cols, 'xbond': [c for c in all_cols if c not in bond_cols]}

yields_svensson = target[[c for c in target.columns if c.startswith('Y')]]
regime_labels = tree_labels['regime_raw'].values
regime_labels_canonical = tree_labels['regime_canonical'].values
regime_series_canonical = tree_labels['regime_canonical']

# Fold trees: n_regimes found by each (test_year, tree_type)
fold_n_regimes = fold_splits.groupby(['test_year', 'tree_type'])['n_regimes'].first()

print(f'Features: {data_q.shape} | Target: {yields_svensson.shape} | Regimes: {np.unique(regime_labels)}')
print(f'Categories: {list(categories)}')

## 1. Helper Functions

In [ ]:
def build_proxy_variables(X_train, Y_train, X_all, categories, random_state=42):
    """
    Proxy-variable construction (Zhao et al. 2025) with sign correction.
      1. Fit a Random Forest regressor X_train -> Y_train (yields).
      2. SHAP importance of each variable = mean |SHAP| over training months and target columns.
      3. Orientation sign = sign of the covariance between the variable and its SHAP contribution
         (+1: higher value pushes yields up, -1: pushes yields down).
      4. Oriented quantile = q if sign > 0 else 1 - q.
      5. Proxy variable of a category = importance-weighted average of its oriented quantiles.
    Returns (proxy variables for X_all, weights table).
    """
    rfr = RandomForestRegressor(n_estimators=n_trees, random_state=random_state, n_jobs=-1)
    rfr.fit(X_train, Y_train)

    shap_values = shap.TreeExplainer(rfr)(X_train).values  # (months, features[, targets])
    if shap_values.ndim == 2:
        shap_values = shap_values[:, :, None]
    importance = np.abs(shap_values).mean(axis=(0, 2))

    X_centered = X_train.values - X_train.values.mean(axis=0)
    covariance = (X_centered * shap_values.mean(axis=2)).mean(axis=0)
    sign = np.where(covariance >= 0, 1, -1)

    weights = pd.DataFrame({'importance': importance, 'sign': sign}, index=X_train.columns)
    weights['pv_category'] = [next(cat for cat, cols in categories.items() if v in cols) for v in weights.index]
    cat_sum = weights.groupby('pv_category')['importance'].transform('sum')
    cat_size = weights.groupby('pv_category')['importance'].transform('size')
    weights['weight'] = np.where(cat_sum > 0, weights['importance'] / cat_sum.replace(0, np.nan), 1 / cat_size)  # equal weights if all zero

    pv = pd.DataFrame(index=X_all.index)
    for cat, group in weights.groupby('pv_category', sort=False):
        oriented = X_all[group.index].copy()
        flip = group.index[group['sign'] < 0]
        oriented[flip] = 1 - oriented[flip]
        pv[cat] = (oriented * group['weight']).sum(axis=1)
    pv = pv[[cat for cat in categories if cat in pv.columns]]  # keep category order
    return pv, weights

def brier_scores(probs, classes, y_true, global_classes):
    """Per-month mean squared error between class probabilities and one-hot truth (columns aligned to global_classes)."""
    aligned = np.zeros((len(y_true), len(global_classes)))
    for j, cls in enumerate(classes):
        aligned[:, global_classes.index(cls)] = probs[:, j]
    one_hot = np.array([[1.0 if y == c else 0.0 for c in global_classes] for y in y_true])
    return np.mean((aligned - one_hot) ** 2, axis=1)

In [ ]:
def shade_regimes(ax, dates, labels, colors, alpha=0.25):
    """Shade contiguous blocks of each regime label (labels may be a Series covering only some dates)."""
    labels = pd.Series(np.asarray(labels), index=dates) if not isinstance(labels, pd.Series) else labels
    for label in sorted(labels.unique()):
        in_block, block_start = False, None
        for date in dates:
            is_label = date in labels.index and labels.loc[date] == label
            if is_label and not in_block:
                in_block, block_start = True, date
            elif not is_label and in_block:
                in_block = False
                ax.axvspan(block_start, date, alpha=alpha, color=colors[label], lw=0, zorder=0)
        if in_block:
            ax.axvspan(block_start, dates[-1] + pd.DateOffset(months=1), alpha=alpha, color=colors[label], lw=0, zorder=0)

def shade_recessions(ax, recession, alpha=0.35):
    in_rec, rec_start = False, None
    for date, val in recession['USREC'].items():
        if val == 1 and not in_rec:
            in_rec, rec_start = True, date
        elif val == 0 and in_rec:
            in_rec = False
            ax.axvspan(rec_start, date, alpha=alpha, color='grey', lw=0, zorder=1)
    if in_rec:
        ax.axvspan(rec_start, recession.index[-1], alpha=alpha, color='grey', lw=0, zorder=1)

## 2. Feature Importance via Random Forest Classification
A Random Forest classifier is trained on the Bayesian-tree labels to see which variables drive each regime (in-sample).

In [ ]:
X = data_q.copy()
y = regime_labels.copy()

base_rf = RandomForestClassifier(n_estimators=n_trees, random_state=random_state).fit(X, y)
y_pred = base_rf.predict(X)
print(f'Random Forest in-sample agreement with Bayesian tree labels: {accuracy_score(y, y_pred):.2%}\n')
print(classification_report(y, y_pred))

plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y, y_pred), annot=True, fmt='d', cmap='Blues', xticklabels=np.unique(y), yticklabels=np.unique(y))
plt.xlabel('Predicted Regime (RF)'); plt.ylabel('Actual Regime (Bayesian Tree)'); plt.title('RF vs Bayesian Tree Labels')
plt.show()

In [ ]:
# SHAP beeswarm per regime, and a waterfall for the middle month of each regime
explainer = shap.TreeExplainer(base_rf)
shap_explanation = explainer(X)

for class_idx in range(len(np.unique(y))):
    print(f'\nBeeswarm — Regime {class_idx}')
    shap.plots.beeswarm(shap_explanation[:, :, class_idx], max_display=20, show=True)

for class_idx in range(len(np.unique(y))):
    regime_months = np.where(y == class_idx)[0]
    sample_idx = regime_months[len(regime_months) // 2]
    print(f'\nWaterfall — Regime {class_idx} at {X.index[sample_idx].date()}')
    shap.plots.waterfall(shap.Explanation(values=shap_explanation.values[sample_idx, :, class_idx],
                                          base_values=explainer.expected_value[class_idx],
                                          data=X.iloc[sample_idx], feature_names=X.columns.tolist()),
                         max_display=20, show=True)

## 3. Dimensionality Reduction via Sign-Corrected Proxy Variables
A Random Forest regressor targeting the NSS yields gives every variable a SHAP importance **and** an orientation sign;
each category's proxy variable is the importance-weighted average of its oriented quantiles.

In [ ]:
proxy_sets, weight_tables = {}, []
for name, cols in feature_sets.items():
    pv, weights = build_proxy_variables(data_q[cols], yields_svensson, data_q[cols], categories)
    proxy_sets[name] = pv
    weight_tables.append(weights.assign(feature_set=name))
    print(f'{name}: {pv.shape[1]} proxy variables from {len(cols)} features')

data_pv = proxy_sets['all']
weights_all = weight_tables[0]
print('\nSHAP importance, orientation and weight of each variable (full sample, all features):')
print(weights_all.sort_values(['pv_category', 'weight'], ascending=[True, False]).round(4).to_string())

In [ ]:
fig, axes = plt.subplots(len(data_pv.columns), 1, figsize=(14, 2.2 * len(data_pv.columns)), sharex=True)
for ax, cat in zip(axes, data_pv.columns):
    ax.plot(common_idx, data_pv[cat], lw=0.9, color='black')
    shade_regimes(ax, common_idx, regime_labels_canonical, regime_colors)
    shade_recessions(ax, recession)
    ax.axhline(0.5, color='gray', lw=0.7, ls='--'); ax.set_ylim(0, 1); ax.set_ylabel(cat, rotation=0, ha='right')
axes[0].set_title('Sign-corrected proxy variables (1 = pushes yields up) shaded by canonical tree regime')
plt.tight_layout(); plt.show()

### 3.1. Benchmark Against PCA and RMT-PCA

The proxy-variable Random Forest (PV-RF) is compared with Random Forests trained on 12 principal components and on 12
components from a Marchenko-Pastur-cleaned correlation matrix (noise eigenvalues below $\lambda_+ = \sigma^2(1+\sqrt{p/T})^2$
replaced by their mean, preserving the trace). Models are compared on Brier score with a Model Confidence Set test.

> NOTE (known issue): all three comparisons below are **in-sample**; an unrestricted forest reproduces its own training
> labels perfectly, so agreement is 100% by construction and the MCS mainly ranks how sharp each forest's leaf probabilities are.

In [ ]:
X_train = data_q[feature_sets['all']]
y_target = regime_labels

# Proxy variables
pv_rf = RandomForestClassifier(n_estimators=n_trees, random_state=random_state, n_jobs=-1).fit(data_pv, y_target)

# PCA (12 components to match the 12 proxy variables)
n_components = data_pv.shape[1]
pca = PCA(n_components=n_components)
X_pca = pd.DataFrame(pca.fit_transform(X_train), index=common_idx, columns=[f'PC{i+1}' for i in range(n_components)])
pca_rf = RandomForestClassifier(n_estimators=n_trees, random_state=random_state, n_jobs=-1).fit(X_pca, y_target)
print(f'PCA explained variance (top {n_components}): {pca.explained_variance_ratio_.sum():.2%}')

# RMT-cleaned PCA
X_std = StandardScaler().fit_transform(X_train)
T, p = X_std.shape
q = p / T
C = X_std.T @ X_std / T
eigenvalues, eigenvectors = np.linalg.eigh(C)
lambda_plus, lambda_minus = (1 + np.sqrt(q)) ** 2, (1 - np.sqrt(q)) ** 2
noise_mask = eigenvalues <= lambda_plus
print(f'T = {T}, p = {p}, q = {q:.4f} | Marchenko-Pastur λ- = {lambda_minus:.4f}, λ+ = {lambda_plus:.4f} | '
      f'signal eigenvalues: {(~noise_mask).sum()}, noise: {noise_mask.sum()}')

eigenvalues_clean = eigenvalues.copy()
eigenvalues_clean[noise_mask] = eigenvalues[noise_mask].mean()
order = np.argsort(eigenvalues_clean)[::-1]
X_rmt = pd.DataFrame(X_std @ eigenvectors[:, order[:n_components]], index=common_idx, columns=[f'RMT_PC{i+1}' for i in range(n_components)])
rmt_rf = RandomForestClassifier(n_estimators=n_trees, random_state=random_state, n_jobs=-1).fit(X_rmt, y_target)
print(f'Variance explained by top {n_components}: PCA {pca.explained_variance_ratio_.sum():.2%} | '
      f'RMT-PCA {eigenvalues_clean[order[:n_components]].sum() / eigenvalues_clean.sum():.2%}')

mp_x = np.linspace(max(lambda_minus, 1e-9), lambda_plus, 500)
mp_pdf = np.sqrt((lambda_plus - mp_x) * (mp_x - lambda_minus)) / (2 * np.pi * q * mp_x)
plt.figure(figsize=(10, 4))
plt.hist(eigenvalues, bins=100, density=True, alpha=0.55, label='Eigenvalues')
plt.plot(mp_x, mp_pdf, color='red', label='Marchenko-Pastur PDF')
plt.axvline(lambda_plus, color='red', ls='--', label=f'λ+ = {lambda_plus:.3f}')
plt.xlabel('Eigenvalue'); plt.ylabel('Density'); plt.title('Eigenvalue Spectrum vs Marchenko-Pastur'); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# In-sample agreement, Brier scores and Model Confidence Set
models = {'PV_RF': (pv_rf, data_pv), 'PCA_RF': (pca_rf, X_pca), 'RMT_PCA_RF': (rmt_rf, X_rmt)}
losses = {}
print('In-sample agreement with Bayesian tree labels / mean Brier score:')
for name, (model, features) in models.items():
    agreement = (model.predict(features) == y_target).mean()
    losses[name] = brier_scores(model.predict_proba(features), model.classes_, y_target, sorted(np.unique(y_target)))
    print(f'  {name:12s} {agreement:6.1%}   Brier = {losses[name].mean():.6f}')

losses_df = pd.DataFrame(losses, index=common_idx)
np.random.seed(random_state)
mcs = MCS(losses_df, size=0.10, method='max', reps=1000)
mcs.compute()
print('\nModel Confidence Set p-values:')
print(mcs.pvalues.sort_values('Pvalue', ascending=False).to_string())

## 4. Expanding-Window Out-of-Sample Evaluation

For every test year Y (12 months) the classifier only sees data up to `gap_months + 1` months before Y:

1. **Local tree** (pre-computed in `TreeClassifier.ipynb`): regime labels on the training window, canonically ordered by mean 10Y-2Y spread.
2. **Proxy variables**: SHAP weights and orientation signs re-estimated on the training window only.
3. **PV-RF** trained on the local canonical labels, predicting the test months.
4. **Ground truth**: either the *evaluation tree* grown on data up to the end of Y (`eval`, the fair benchmark) or the
   full-sample tree (`full`, which uses regime definitions shaped by later data).

`forecast=True` turns this into the 1-month-ahead forecaster: the regressor and classifier are trained on features at
$t$ against yields / regime at $t+1$, and test-month regimes are predicted from features observed before the gap month.

> TODO (known issue, not fixed yet): the forecaster trains at a 1-month horizon but predicts test months from features
> `1 + gap_months` = 2 months earlier, so training and test horizons differ. Kept as in MacroRegimes2 for now.

In [ ]:
def run_expanding_window(feature_set='all', ground_truth='eval', forecast=False):
    """Expanding-window PV-RF evaluation. Returns (monthly results DataFrame, SHAP importance per fold DataFrame)."""
    cols = feature_sets[feature_set]
    local_type = f'local_{feature_set}'
    results, shap_history = [], []

    first_test_year = (common_idx.min() + pd.DateOffset(years=min_training_years)).year
    for test_year in range(first_test_year, common_idx.max().year + 1):
        test_start, test_end = pd.Timestamp(f'{test_year}-01-01'), pd.Timestamp(f'{test_year}-12-01')
        test_idx = common_idx[(common_idx >= test_start) & (common_idx <= test_end)]
        local = fold_labels[(fold_labels['test_year'] == test_year) & (fold_labels['tree_type'] == local_type)].set_index('date')
        train_idx = local.index

        if len(test_idx) == 0 or len(train_idx) < min_training_years * 12:
            print(f'[{test_year}] insufficient training data, skipping.')
            continue
        if fold_n_regimes.get((test_year, local_type), 0) < max_regimes:
            print(f'[{test_year}] local tree found < {max_regimes} regimes, skipping.')
            continue
        if ground_truth == 'eval':
            if fold_n_regimes.get((test_year, 'eval'), 0) < max_regimes:
                print(f'[{test_year}] evaluation tree found < {max_regimes} regimes, skipping.')
                continue
            truth = fold_labels[(fold_labels['test_year'] == test_year) & (fold_labels['tree_type'] == 'eval')].set_index('date')
            y_test_true = truth.loc[test_idx, 'regime_canonical'].values
        else:
            y_test_true = regime_series_canonical.loc[test_idx].values

        local_canonical = local['regime_canonical'].values

        # Training pairs: contemporaneous (X_t -> regime_t) or forecasting (X_t -> regime_{t+1})
        if forecast:
            X_train_fold = data_q.loc[train_idx, cols].iloc[:-1]
            Y_train_fold = yields_svensson.loc[train_idx].iloc[1:]
            y_train_fold = local_canonical[1:]
            test_feature_idx = test_idx - pd.DateOffset(months=1 + gap_months)
        else:
            X_train_fold = data_q.loc[train_idx, cols]
            Y_train_fold = yields_svensson.loc[train_idx]
            y_train_fold = local_canonical
            test_feature_idx = test_idx

        pv_fold, weights_fold = build_proxy_variables(X_train_fold, Y_train_fold, data_q[cols], categories)
        shap_history.append(weights_fold['importance'].rename(test_year))

        clf = RandomForestClassifier(n_estimators=n_trees, random_state=random_state, n_jobs=-1)
        clf.fit(pv_fold.loc[X_train_fold.index], y_train_fold)
        pred = clf.predict(pv_fold.loc[test_feature_idx])
        probs = clf.predict_proba(pv_fold.loc[test_feature_idx])

        unseen = set(np.unique(y_test_true)) - set(np.unique(y_train_fold))
        if unseen:
            print(f'[{test_year}] warning: regime(s) {unseen} in test period never seen in training.')

        brier = brier_scores(probs, clf.classes_, y_test_true, global_classes)
        acc = (pred == y_test_true).mean()
        print(f'[{test_year}] train {train_idx[0].date()} → {train_idx[-1].date()} ({len(train_idx)} months) | '
              f'accuracy {acc:6.1%} | Brier {brier.mean():.4f}')

        for i, date in enumerate(test_idx):
            results.append({'date': date, 'test_year': test_year, 'true_regime': y_test_true[i], 'pred_regime': pred[i],
                            'correct': int(y_test_true[i] == pred[i]), 'brier': brier[i]})

    results = pd.DataFrame(results).set_index('date')
    print('\n' + '=' * 60)
    print(f'TOTAL OOS MONTHS: {len(results)} | ACCURACY: {results["correct"].mean():.2%} | BRIER: {results["brier"].mean():.6f}')
    print('=' * 60)
    print(results.groupby('test_year')[['correct', 'brier']].mean().round(4).to_string())
    return results, pd.concat(shap_history, axis=1).T

In [ ]:
def transition_analysis(results):
    """Did the model flag regime transitions when they happened (recall), and how often did it cry wolf (precision)?"""
    truth, pred = results['true_regime'], results['pred_regime']
    tp, fn, fp, transition_dates = [], [], [], []
    for date in results.index.sort_values():
        prev = date - pd.DateOffset(months=1)
        if prev not in results.index:
            continue
        true_change, pred_change = truth[date] != truth[prev], pred[date] != pred[prev]
        if true_change and pred_change:
            tp.append({'date': date, 'from': truth[prev], 'to': truth[date], 'pred_from': pred[prev], 'pred_to': pred[date],
                       'correct': pred[date] == truth[date]}); transition_dates.append(date)
        elif true_change:
            fn.append({'date': date, 'from': truth[prev], 'to': truth[date], 'pred': pred[date]}); transition_dates.append(date)
        elif pred_change:
            fp.append({'date': date, 'true': truth[date], 'pred_from': pred[prev], 'pred_to': pred[date]})

    n_tp, n_fn, n_fp = len(tp), len(fn), len(fp)
    n_correct = sum(t['correct'] for t in tp)
    recall = n_tp / (n_tp + n_fn) if n_tp + n_fn else np.nan
    precision = n_tp / (n_tp + n_fp) if n_tp + n_fp else np.nan
    non_transition_acc = results.loc[~results.index.isin(transition_dates), 'correct'].mean()

    print('=' * 60 + '\nREGIME TRANSITION ANALYSIS\n' + '=' * 60)
    print(f'Total months: {len(results)} | actual transition months: {n_tp + n_fn}')
    print(f'True positives (detected):     {n_tp}  ({recall:.1%} recall)')
    print(f'False negatives (missed):      {n_fn}')
    print(f'False positives (false alarm): {n_fp}  ({precision:.1%} precision)')
    print(f'Detected transitions correctly classified: {n_correct}/{n_tp}')
    print(f'All transitions correctly classified:      {n_correct}/{n_tp + n_fn}')
    print(f'Non-transition accuracy: {non_transition_acc:.1%}')
    for title, rows in [('True positives', tp), ('False negatives (missed transitions)', fn), ('False positives (false alarms)', fp)]:
        if rows:
            print(f'\n{title}:')
            print(pd.DataFrame(rows).set_index('date').to_string())

def shap_stability(shap_history, title, top_n=10):
    """Evolution of SHAP importances across folds, rank heatmap, and mean Spearman rank correlation between consecutive folds."""
    top_features = shap_history.mean().sort_values(ascending=False).head(top_n).index

    plt.figure(figsize=(12, 5))
    for col in top_features:
        plt.plot(shap_history.index, shap_history[col], marker='o', label=col, lw=1)
    plt.title(f'Evolution of Top {top_n} SHAP Weights — {title}'); plt.xlabel('Test Year'); plt.ylabel('Mean |SHAP|')
    plt.legend(loc='best', fontsize=8); plt.grid(True, ls='--', alpha=0.3); plt.tight_layout(); plt.show()

    ranks = shap_history.rank(axis=1, ascending=False)
    plt.figure(figsize=(14, 6))
    sns.heatmap(ranks[top_features].T, annot=True, cmap='RdYlGn_r', fmt='.0f')
    plt.title(f'Feature Rank Stability (1 = most important) — {title}'); plt.xlabel('Test Year'); plt.tight_layout(); plt.show()

    corrs = [shap_history.iloc[i].corr(shap_history.iloc[i + 1], method='spearman') for i in range(len(shap_history) - 1)]
    print(f'Average Spearman rank correlation between consecutive folds: {np.mean(corrs):.4f}')

### 4.1. OOS Classification (all features, evaluation-tree ground truth)

In [ ]:
results_oos, shap_oos = run_expanding_window('all', ground_truth='eval')

In [ ]:
transition_analysis(results_oos)

In [ ]:
shap_stability(shap_oos, 'OOS classifier')

### 4.2. OOS Classification Against the Full-Sample Tree
Full-sample labels are only an evaluation benchmark here (never used for training), analogous to scoring a real-time
recession indicator against NBER dates that are only announced later. Expect worse numbers: the full-sample tree's regime
definitions are shaped by data (e.g. the 2022+ hiking cycle) that a model trained through 2015 could not have seen.

In [ ]:
results_oos_full, _ = run_expanding_window('all', ground_truth='full')

In [ ]:
transition_analysis(results_oos_full)

### 4.3. OOS Classification Without Bond-Market Variables
Because the tree's target is the yield curve, yield variables (GS10 above all) have an inherent advantage. Removing the
bond-market category from both the local tree and the classifier tests whether real-economy variables alone recover the
regime structure. The evaluation tree still uses all variables.

In [ ]:
results_oos_xbond, shap_xbond = run_expanding_window('xbond', ground_truth='eval')

In [ ]:
transition_analysis(results_oos_xbond)

In [ ]:
shap_stability(shap_xbond, 'OOS classifier without bond market')

### 4.4. 1-Month-Ahead Forecaster

In [ ]:
results_fcast, shap_fcast = run_expanding_window('all', ground_truth='eval', forecast=True)

In [ ]:
transition_analysis(results_fcast)

In [ ]:
shap_stability(shap_fcast, 'forecaster (t → t+1)')

## 5. Compiled Results

In [ ]:
proxy_sets['all'].to_csv(f'{data_folder}/MacroRegimes3_PVRF_ProxyVariables.csv', index_label='date')
proxy_sets['xbond'].to_csv(f'{data_folder}/MacroRegimes3_PVRF_ProxyVariables_xbond.csv', index_label='date')
pd.concat(weight_tables).to_csv(f'{data_folder}/MacroRegimes3_PVRF_ProxyWeights.csv', index_label='variable')
results_oos.to_csv(f'{data_folder}/MacroRegimes3_PVRF_OOS_Results.csv')
results_oos_full.to_csv(f'{data_folder}/MacroRegimes3_PVRF_OOS_FullTruth_Results.csv')
results_oos_xbond.to_csv(f'{data_folder}/MacroRegimes3_PVRF_OOS_xbond_Results.csv')
results_fcast.to_csv(f'{data_folder}/MacroRegimes3_PVRF_FCAST_Results.csv')
print('Saved proxy variables, weights and all OOS result files.')

In [ ]:
plot_maturities = {'Y3M': '3M', 'Y6M': '6M', 'Y12M': '1Y', 'Y24M': '2Y', 'Y60M': '5Y', 'Y120M': '10Y'}
line_colors = ['#4575b4', '#74add1', '#abd9e9', '#fdae61', '#f46d43', '#d73027']
panels = [('Full-Sample Regimes — Bayesian Tree', regime_series_canonical),
          ('OOS Classifier — PV-RF Predicted Regimes (training period unshaded)', results_oos['pred_regime']),
          ('OOS Forecaster — PV-RF Predicted Regimes (training period unshaded)', results_fcast['pred_regime'])]

fig, axes = plt.subplots(3, 1, figsize=(16, 15))
for ax, (title, series) in zip(axes, panels):
    for (col, label), lc in zip(plot_maturities.items(), line_colors):
        ax.plot(common_idx, yields_svensson[col], color=lc, lw=1.2, label=label, zorder=3)
    shade_regimes(ax, common_idx, series, regime_colors)
    shade_recessions(ax, recession)
    handles = [plt.Line2D([0], [0], color=lc, lw=1.5, label=label) for lc, label in zip(line_colors, plot_maturities.values())]
    handles += [plt.matplotlib.patches.Patch(color=regime_colors[r], alpha=0.5, label=f'Regime {r} ({(series == r).sum()} months)') for r in sorted(series.unique())]
    handles += [plt.matplotlib.patches.Patch(color='grey', alpha=0.4, label='NBER Recession')]
    ax.legend(handles=handles, loc='upper right', ncol=3, fontsize=8)
    ax.set_title(title, fontsize=11, fontweight='bold'); ax.set_ylabel('Yield (%)'); ax.grid(True, alpha=0.25)
    ax.set_xlim(common_idx.min() - pd.DateOffset(months=1), common_idx.max() + pd.DateOffset(months=1))
axes[-1].set_xlabel('Date')
plt.tight_layout(); plt.show()